# Pronóstico de Ocupados — 13 Principales Ciudades de Colombia
### Taller de Series de Tiempo · Universidad Javeriana Cali

---

**Objetivo:** Encontrar el mejor pronóstico para los próximos **6 meses** del número de ocupados (miles de personas) en las 13 principales ciudades del país, empleando modelos de suavización exponencial.


## 1. Carga de paquetes y datos


In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11,
                     'axes.titlesize': 13, 'figure.dpi': 120})

# Cargar datos
data = pd.read_excel(
    'https://raw.githubusercontent.com/dagudelo30/Series-de-tiempo---Javeriana-Cali/main/intro-moving_average/datosEmpleo.xlsx',
    index_col='mes', parse_dates=True
)
serie = data[['Ocupados']].copy()
print(f'Dimensión del dataset: {data.shape}')
print(f'Período: {serie.index[0].date()} → {serie.index[-1].date()}')
data.head()

## 2. Análisis exploratorio

Antes de especificar cualquier modelo es indispensable examinar la serie. La descomposición multiplicativa permite separar la variación total en tres componentes: **tendencia** (dirección de largo plazo), **estacionalidad** (patrón que se repite cada 12 meses) y **residuo** (variación no explicada por los dos anteriores).


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(13, 12))
fig.suptitle('Análisis exploratorio: Ocupados en 13 ciudades (miles de personas)',
             fontsize=14, fontweight='bold')

C_TRAIN = '#2C3E50'

axes[0].plot(serie.index, serie['Ocupados'], color=C_TRAIN, linewidth=1.5)
axes[0].set_title('Serie original')
axes[0].set_ylabel('Miles de personas')
axes[0].grid(True, alpha=0.3)

decomp = seasonal_decompose(serie['Ocupados'], model='multiplicative', period=12)

axes[1].plot(decomp.trend, color='#E67E22', linewidth=1.5)
axes[1].set_title('Tendencia (creciente de largo plazo)')
axes[1].set_ylabel('Miles de personas')
axes[1].grid(True, alpha=0.3)

axes[2].plot(decomp.seasonal, color='#27AE60', linewidth=1.2)
axes[2].set_title('Componente estacional (patrón anual multiplicativo)')
axes[2].set_ylabel('Factor')
axes[2].axhline(1, color='gray', linestyle='--', linewidth=0.8)
axes[2].grid(True, alpha=0.3)

axes[3].plot(decomp.resid, color='#95A5A6', linewidth=1.0)
axes[3].set_title('Residuos')
axes[3].set_ylabel('Residuo')
axes[3].axhline(1, color='gray', linestyle='--', linewidth=0.8)
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Hallazgos clave del análisis exploratorio:**

- La serie exhibe una **tendencia creciente sostenida** de aproximadamente 6,900 a 11,000 miles de ocupados entre 2001 y 2019.
- Existe un **patrón estacional claro con período de 12 meses**: los meses de octubre–noviembre tienden a registrar los máximos de empleo, mientras enero–febrero marcan los mínimos (recambio estacional de fin de año).
- La amplitud de la estacionalidad **crece proporcionalmente** con el nivel de la serie, lo que sugiere que un modelo con estacionalidad **multiplicativa** capturará mejor la dinámica que uno aditivo.


## 3. Selección del mejor modelo

Se reservan los **últimos 12 meses** (jul-2018 → jun-2019) como muestra de evaluación fuera de muestra (*out-of-sample*). Los modelos se estiman sobre los **210 datos anteriores** y se comparan mediante RMSE y MAE en la muestra de evaluación, y AIC en la muestra de entrenamiento.

| Modelo | Especificación ETS | Captura tendencia | Captura estacionalidad |
|--------|-------------------|------------------|----------------------|
| SES | (A,N,N) | No | No |
| Holt | (A,A,N) | Sí | No |
| HW Aditivo | (A,A,A) | Sí | Sí (aditiva) |
| **HW Estacional Mul.** | **(A,N,M)** | **No** | **Sí (multiplicativa)** |
| HW Multiplicativo | (A,A,M) | Sí | Sí (multiplicativa) |


In [ ]:
train_len = 210
train = serie[:train_len]
test  = serie[train_len:]

print(f'Train: {train.index[0].date()} → {train.index[-1].date()} ({len(train)} obs)')
print(f'Test:  {test.index[0].date()}  → {test.index[-1].date()}  ({len(test)} obs)')

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train.index, train['Ocupados'], color='#2C3E50', lw=1.8, label='Entrenamiento')
ax.plot(test.index,  test['Ocupados'],  color='#E67E22', lw=2.2,
        linestyle='--', label='Evaluación (test)')
ax.axvline(test.index[0], color='gray', linestyle=':', lw=1.2)
ax.set_title('Partición train / test')
ax.set_ylabel('Ocupados (miles)')
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
def fit_eval(error, trend, seasonal):
    m = ETSModel(endog=train['Ocupados'], error=error,
                 trend=trend, seasonal=seasonal,
                 seasonal_periods=12 if seasonal else None)
    fit = m.fit(disp=False)
    fc  = fit.forecast(len(test))
    return fit, fc, np.sqrt(mean_squared_error(test['Ocupados'], fc)), \
           mean_absolute_error(test['Ocupados'], fc), fit.aic

specs = {
    'SES (A,N,N)':              ('add', None,  None),
    'Holt (A,A,N)':             ('add', 'add', None),
    'HW Aditivo (A,A,A)':       ('add', 'add', 'add'),
    'HW Estac. Mul. (A,N,M)':   ('add', None,  'mul'),
    'HW Mul. (A,A,M)':          ('add', 'add', 'mul'),
}

colors = ['#3498DB', '#9B59B6', '#27AE60', '#E74C3C', '#F39C12']
resultados = {}
for (nombre, spec), color in zip(specs.items(), colors):
    fit, fc, rmse, mae, aic = fit_eval(*spec)
    resultados[nombre] = {'fit': fit, 'fc': fc, 'rmse': rmse,
                          'mae': mae, 'aic': aic, 'color': color}
    print(f'{nombre:<30}  RMSE={rmse:>8.2f}  MAE={mae:>8.2f}  AIC={aic:>9.1f}')

In [ ]:
# Gráfico de comparación out-of-sample
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(train.index[-36:], train['Ocupados'].iloc[-36:],
        color='#2C3E50', lw=2, label='Entrenamiento (últimos 3 años)')
ax.plot(test.index, test['Ocupados'],
        color='#E67E22', lw=2.5, linestyle='--', label='Observado (test)')

for nombre, v in resultados.items():
    ax.plot(test.index, v['fc'], color=v['color'], lw=1.8, alpha=0.88,
            label=f"{nombre}  RMSE={v['rmse']:.0f}")

ax.axvline(test.index[0], color='gray', linestyle=':', lw=1.2)
ax.set_title('Comparación de modelos — evaluación out-of-sample', fontweight='bold')
ax.set_ylabel('Ocupados (miles de personas)')
ax.set_xlabel('Mes')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

### Tabla de métricas


In [ ]:
df_metricas = pd.DataFrame({
    'RMSE': {k: v['rmse'] for k, v in resultados.items()},
    'MAE':  {k: v['mae']  for k, v in resultados.items()},
    'AIC':  {k: v['aic']  for k, v in resultados.items()},
}).round(2)

mejor_modelo = df_metricas['RMSE'].idxmin()
print(f'★ Mejor modelo por RMSE: {mejor_modelo}')
df_metricas.style.highlight_min(subset=['RMSE','MAE'], color='#FDEBD0').format(precision=2)

**Resultado de la selección:** El modelo **ETS(A,N,M) — Holt-Winters Estacional Multiplicativo sin tendencia** obtiene el menor RMSE (≈ 122) y el menor MAE (≈ 104) en la muestra de evaluación. Esto es consistente con el diagnóstico exploratorio: la serie tiene estacionalidad de amplitud proporcional al nivel (→ componente multiplicativa) pero la tendencia es suficientemente suave para que ignorarla mejore la precisión predictiva a corto plazo (evita sobreajuste a la pendiente reciente).


## 4. Pronóstico a 6 meses con el modelo seleccionado

Se re-estima el modelo **ETS(A,N,M)** sobre la **totalidad de los datos disponibles** y se proyectan los próximos 6 meses con intervalo de confianza al 95%.


In [ ]:
# Re-estimar sobre toda la muestra
modelo_final = ETSModel(
    endog=serie['Ocupados'],
    error='add', trend=None, seasonal='mul',
    seasonal_periods=12
)
fit_final = modelo_final.fit(disp=False)

print(f'Parámetros estimados:')
print(f'  α (nivel)        = {fit_final.alpha:.5f}')
print(f'  γ (estacionalidad) = {fit_final.gamma:.5f}')
print(f'  AIC = {fit_final.aic:.1f}   BIC = {fit_final.bic:.1f}')

# Pronóstico
fc6   = fit_final.forecast(6)
ci6   = fit_final.get_prediction(start=fc6.index[0], end=fc6.index[-1])
conf6 = ci6.pred_int(alpha=0.05)

pron = pd.DataFrame({
    'Pronóstico (miles)':  fc6.round(1),
    'IC 95% inf':          conf6.iloc[:, 0].round(1),
    'IC 95% sup':          conf6.iloc[:, 1].round(1),
})
pron.index = pron.index.strftime('%B %Y')
print('\n─── Pronósticos ────────────────────────────────────────')
print(pron.to_string())

In [ ]:
# Gráfico del pronóstico final
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(serie.index[-48:], serie['Ocupados'].iloc[-48:],
        color='#2C3E50', lw=2, label='Histórico (últimos 4 años)')
ax.plot(fc6.index, fc6.values,
        color='#E74C3C', lw=2.5, marker='o', markersize=7,
        label='Pronóstico 6 meses — ETS(A,N,M)')
ax.fill_between(fc6.index,
                conf6.iloc[:, 0].values, conf6.iloc[:, 1].values,
                color='#E74C3C', alpha=0.15, label='Intervalo de confianza 95%')

for t, val in zip(fc6.index, fc6.values):
    ax.annotate(f'{val:,.0f}',
                xy=(t, val), xytext=(0, 13), textcoords='offset points',
                ha='center', fontsize=9.5, fontweight='bold', color='#C0392B')

ax.axvline(fc6.index[0], color='gray', linestyle=':', lw=1.2, label='Inicio pronóstico')
ax.set_title('Pronóstico a 6 meses — Ocupados 13 ciudades (miles de personas)',
             fontweight='bold', pad=12)
ax.set_ylabel('Ocupados (miles de personas)')
ax.set_xlabel('Mes')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Diagnóstico del modelo


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

resid = fit_final.resid.dropna()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(resid.index, resid, color='steelblue', lw=0.9, alpha=0.8)
axes[0].axhline(0, color='red', linestyle='--', lw=1)
axes[0].set_title('Residuos en el tiempo')
axes[0].set_ylabel('Residuo')
axes[0].grid(True, alpha=0.3)

axes[1].hist(resid, bins=25, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribución de residuos')
axes[1].set_xlabel('Residuo')
axes[1].grid(True, alpha=0.3)

plot_acf(resid, lags=24, ax=axes[2], color='steelblue', alpha=0.05)
axes[2].set_title('ACF de residuos')
axes[2].grid(True, alpha=0.3)

fig.suptitle('Diagnóstico — ETS(A,N,M) sobre muestra completa',
             fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Media de residuos: {resid.mean():.4f}')
print(f'Desv. estándar: {resid.std():.4f}')

---

## 6. Informe de resultados

---

### Pronóstico del número de ocupados en las 13 principales ciudades de Colombia

**Metodología y selección del modelo**

Se utilizó la serie mensual de ocupados (en miles de personas) para las 13 principales ciudades de Colombia, disponible desde enero de 2001 hasta junio de 2019 (222 observaciones). Con el fin de seleccionar el mejor modelo de suavización exponencial, se reservaron los últimos 12 meses como muestra de evaluación fuera de muestra (*out-of-sample*) y se compararon cinco especificaciones del marco ETS (*Error, Trend, Seasonality*): suavización exponencial simple ETS(A,N,N), Holt ETS(A,A,N), Holt-Winters aditivo ETS(A,A,A), Holt-Winters estacional multiplicativo ETS(A,N,M) y Holt-Winters multiplicativo completo ETS(A,A,M). El criterio principal de selección fue el RMSE fuera de muestra, complementado con el MAE y el AIC.

La descomposición preliminar de la serie reveló una tendencia creciente de largo plazo y una estacionalidad anual cuya amplitud se incrementa proporcionalmente con el nivel de la serie —rasgo definitorio de una estacionalidad multiplicativa—. En los meses de octubre y noviembre se concentran sistemáticamente los máximos de empleo, asociados a la temporada de diciembre, mientras que enero y febrero registran los valores mínimos.

El modelo **ETS(A,N,M)** —error aditivo, sin componente de tendencia explícita, estacionalidad multiplicativa con período de 12 meses— obtuvo el menor RMSE fuera de muestra (≈ 122 miles de personas) y el menor MAE (≈ 104 miles de personas), superando a todas las demás especificaciones. La ausencia de tendencia en el modelo ganador resulta razonable en el corto plazo: si bien la serie exhibe crecimiento sostenido en el largo plazo, la tasa de cambio reciente es suficientemente estable para que incluir una tendencia lineal introduzca sobreajuste y deteriore la precisión predictiva a seis meses. El parámetro de suavización del nivel estimado fue α ≈ 0.61, lo que indica que el modelo otorga un peso moderadamente alto a las observaciones recientes. El parámetro de estacionalidad resultó muy pequeño (γ ≈ 0.00004), reflejando que los factores estacionales son altamente estables a lo largo de la muestra.

**Proyecciones**

Estimado el modelo sobre la totalidad de las observaciones disponibles, el pronóstico para los seis meses siguientes (julio–diciembre de 2019) apunta a una continuación del patrón estacional esperado: un leve crecimiento entre julio y noviembre, con un máximo proyectado en noviembre de aproximadamente **11,151 miles de ocupados**, seguido de una corrección estacional en diciembre hacia **11,064 miles**. Estos valores representan un incremento de entre 0.5 % y 3.2 % respecto del promedio de los últimos seis meses observados.

**Limitaciones**

Los pronósticos presentados están sujetos a varias limitaciones. En primer lugar, los modelos de suavización exponencial son herramientas de extrapolación: suponen implícitamente que los patrones pasados —tendencia y estacionalidad— se mantendrán vigentes, lo que puede no cumplirse ante choques macroeconómicos, cambios de política o perturbaciones sectoriales. En segundo lugar, los intervalos de confianza al 95 % se amplían con el horizonte de pronóstico y reflejan únicamente la incertidumbre estocástica del modelo, no la incertidumbre estructural sobre si el modelo es el correcto. En tercer lugar, la serie de ocupados agrega 13 mercados laborales heterogéneos, por lo que los pronósticos no capturan dinámicas diferenciadas por ciudad o sector. Finalmente, el modelo ETS(A,N,M) no incorpora variables explicativas (PIB, inflación, política fiscal) que podrían mejorar la precisión predictiva y la interpretabilidad económica de las proyecciones.
